In [262]:
import torch
from torch import nn, Tensor
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
import json
import random
from sklearn.preprocessing import StandardScaler
from numpy import log
import math
import numpy
THRESHOLD_TIMESTAMPS = 16
L_SEQUENCE_LENGHT = 64
EMBEDDING_DIM = 32

EPOCHS = 10

In [ ]:
def extract_json(filename: str):
    """
    Extracts the json that are used in the project (OttoDataset) "train.jsonl"
    - input -> filename : the name of the session
    - return all the sessions
    """
    with open(filename, "r") as f:
        for line in f:
            session = json.loads(line)
            yield session["session"], session["events"]

In [264]:
sessions_bf_threshold = []

for i, (session_id, eventstotal) in enumerate(extract_json("../train.jsonl")):
    aids, timestamps, events_type = [], [], []
    for event in eventstotal:
        aids.append(event["aid"])
        timestamps.append(event["ts"])
        events_type.append(event["type"])
        
    sessions_bf_threshold.append({
            "session_id": i,
            "events": {
            "aid": aids,
            "timestamps": timestamps,
            "events_type": events_type    
            },
        })
    if i >= 200:
        break

In [ ]:
class OttoDataSetSession(Dataset):
    """
    The first Dataset, before the TRACE Paper Cut Threshold of the Timestamps which in this case is 16 (THRESHOLD_TIMESTAMPS) 
    It Transforms the click aids in numbers 
    """
    def __init__(self, session):
        self.session = session
        self.event_map = {"clicks":1, "carts": 2, "orders": 3}

    def __len__(self) -> int:
        return len(self.session)

    def __getitem__(self, index):
        session = self.session[index]
                 
        events = session["events"]
        
        aids = torch.tensor(events["aid"], dtype=torch.int64)
        
        timestamps = torch.tensor(events["timestamps"], dtype=torch.long)
        
        events_type = torch.tensor( [self.event_map[e] for e in events["events_type"]], dtype=torch.int64)
        return {
            "session_id": torch.tensor(session["session_id"], dtype=torch.int64),
            "aid": aids,
            "timestamps": timestamps,
            "type": events_type
        }
    

In [266]:
sessions_in_dataset = OttoDataSetSession(sessions_bf_threshold)
print(f"Total len of the Sessions: {len(sessions_in_dataset)}")

session_sample_lenght_after_threshold = []
for i in range(len(sessions_in_dataset)):
    sample = sessions_in_dataset[i]
    if len(sample["timestamps"]) >= THRESHOLD_TIMESTAMPS:
        session_sample_lenght_after_threshold.append(sample)

#print(np.mean(session_sample_lenght_after_threshold))
#print(f"The total of the median is for this total of {len(session_sample_lenght_after_threshold)} {np.median(session_sample_lenght_after_threshold)}")

Total len of the Sessions: 201


In [ ]:

def most_frequent(List):
    """
    
    """
    dict = {}
    count, itm = 0, ''
    for item in reversed(List):
        dict[item] = dict.get(item, 0) + 1
        if dict[item] >= count :
            count, itm = dict[item], item
    return(count, itm)

def custom_collate(batch):
    """
    aids = [
    torch.tensor([aid_to_idx[a] for a in d["aid"]], dtype=torch.long)
    for d in batch
    ]
    """
    aids = [torch.tensor(d["aid"]) for d in batch]
    
    timestamps = [torch.tensor(d["timestamps"]) for d in batch]
    events_type = [torch.tensor(d["type"]) for d in batch]
    
    
    
    
    aids_padded = pad_sequence(aids, batch_first=True, padding_value=0)
    timestamps_padded = pad_sequence(timestamps, batch_first=True, padding_value=0)
    events_type_padded = pad_sequence(events_type, batch_first=True, padding_value=0)
    
    session_id = [d["session_id"] for d in batch]
    return {
        "aid" : aids_padded,
        "timestamps" : timestamps_padded,
        "events_type" : events_type_padded,
        "session_id" : torch.stack(session_id)
    }


In [ ]:
class CutOttoDataSet(OttoDataSetSession):
    def __init__(self, session):
        super().__init__(session)
        self.divide_train,self.divide_test = self.__cut_input_target__()
        self.train = self.divide_train
        self.test = self.divide_test
        
    def __getitem__(self, index):
        session = self.session[index]

        return {
            "session_id": session["session_id"],
            "aid": session["aid"],
            "timestamps": session["timestamps"],
            "type": session["type"]
        }
     
        
    
    def __padding__(self, session):   
        padd_len = L_SEQUENCE_LENGHT - len(session["timestamps"])
        zeros = torch.zeros(padd_len, dtype=session["aid"].dtype)
        
        aid_padded = torch.concat((session["aid"], zeros))
        timestamps_padded = torch.concat((session["timestamps"], zeros))
        type_padded = torch.concat((session["type"], zeros))
        return {
            "session_id":session["session_id"],
            "aid": aid_padded,
            "timestamps": timestamps_padded,
            "type": type_padded
        }
           
        
    def __cut_input_target__(self, min_value=0.80, max_value=0.90):
        input_batches = []
        target_batches = []

        for single_session in self.session:
            cutting_number = random.uniform(min_value, max_value)

            
            n_events = len(single_session["timestamps"])
            train_size = int(n_events * cutting_number)

            input_part = {
                "session_id": single_session["session_id"],
                "aid": single_session["aid"][:train_size],
                "timestamps": single_session["timestamps"][:train_size],
                "type": single_session["type"][:train_size]
            }

            target_part = {
                "session_id": single_session["session_id"],
                "aid": single_session["aid"][train_size:],
                "timestamps": single_session["timestamps"][train_size:],
                "type": single_session["type"][train_size:]
            }

            input_batches.append(input_part)
            target_batches.append(target_part)

        return input_batches, target_batches


            
    def __sequenceLenght__(self):
        sessions_after_sequence_lenght = []
        for session in self.train:
            if len(session["timestamps"]) >= L_SEQUENCE_LENGHT:
                sessions_after_sequence_lenght.append({
                "session_id": session["session_id"],
                "aid": session["aid"][:L_SEQUENCE_LENGHT],
                "timestamps": session["timestamps"][:L_SEQUENCE_LENGHT],
                "type": session["type"][:L_SEQUENCE_LENGHT]
            })
            else: 
                sessions_after_sequence_lenght.append(self.__padding__(session))
        return sessions_after_sequence_lenght
    
    
    """
    Logis of the the model
    """
    def __ATC_task_logit__(self):
        """
        Labels for ATC (User added to the cart in the FUTURE)
        """
        labels_atc = {}

        #Index future_test
        test_by_id = {
            int(session["session_id"]): session
            for session in self.test
        }

        
        for train_session in self.train:
            session_id = int(train_session["session_id"])

            
            test_session = test_by_id.get(session_id, None)

            if 3 in test_session["type"]:
                labels_atc[session_id] = 1
            else:
                labels_atc[session_id] = 0

        return labels_atc
    
    def __SAT__task_logit__(self):
        """
        Counts the highest number of timestamps per product, used in Logit SAT4 (Seeing the same Aid 4 times)
        """
        
        logits_SAT = []
        for session in self.test:
            AidsS_repeated = []
            count = 0    
            for aids in session["aid"]:
                AidsS_repeated.append(aids.item())
                count, product = most_frequent(AidsS_repeated)
            if count >= 4:
                logits_SAT.append(1)
            else:
                logits_SAT.append(0)
        return logits_SAT
    
    def __PD1_task_logit___(self):
        """
        Logits for PD1(Make any Purchase within a day)
        """
        logits_PD1 = []
        ONE_DAY = (86400 * 1000)
        for session in self.test:
            last_ts = session["timestamps"][-1].item()
            ordered_ts = session["timestamps"][session["type"] == 3]
            #Convert into int
            orders_ts = [ts.item() for ts in ordered_ts]
            
            is_purchase = any([(order <= last_ts + ONE_DAY) for order in orders_ts] )
            if is_purchase == True:
                logits_PD1.append(1)
            else:
                logits_PD1.append(0)
            
        return logits_PD1
    
    def __RA1_task_logit___(self):
        """
        Logits for RA1(Return to the same Aid in 1 days)
        """
        ONE_DAY = (86400 * 1000) 
        logits_RA1 = []
        for session in self.test:
            first_aid = session["aid"][0].item()
            first_ts = session["timestamps"][0].item()
            indices = [index for index, aid in enumerate(session["aid"]) if aid.item() == first_aid]
            other_ts_list = session["timestamps"][indices[1:]]

            is_aids = any((other_ts - first_ts) <= ONE_DAY for other_ts in other_ts_list)
            
            if is_aids == True:
                logits_RA1.append(1)
            else: 
                logits_RA1.append(0)
        return logits_RA1
            
           


In [ ]:
#DataSet    
data_set_after_L = CutOttoDataSet(session_sample_lenght_after_threshold)
        
#Training and testing batch size
input_set, target_set = data_set_after_L.__cut_training_testing__()
      
#See how many lenghts are for the Aids in the raining batch and testing batch
print(f"Input Set Len: {len(input_set[0]["aid"])}")
print(f"Target Len: {len(target_set[0]["aid"])}")
    
print(len(data_set_after_L.__sequenceLenght__()[0]["timestamps"]))
    
print("================================================ (Logits part) ===================================================")
print("Logits for the ATC (Add to the Cart)")
print(data_set_after_L.__ATC_task_logit__())
    
print("Logits for SAT4(Seeing the same Aid 4 times)")
print(data_set_after_L.__SAT__task_logit__())
    
print("Logits for PD1(Make any Purchase within a day)")
print(data_set_after_L.__PD1_task_logit___())
    
print("Logits for RA1(Return to the same Aid in 1 days)")
print(data_set_after_L.__RA1_task_logit___())

Training Batch Len: 246
Testing Batch Len: 30
64
================================================ (Logits part) ===================================================
Logits for the ATC (Add to the Cart)
{0: 1, 1: 0, 2: 0, 3: 1, 4: 0, 6: 0, 7: 0, 11: 0, 13: 1, 14: 1, 15: 1, 17: 0, 18: 0, 19: 1, 21: 0, 23: 0, 24: 0, 25: 0, 26: 1, 28: 0, 29: 0, 30: 0, 32: 0, 33: 0, 34: 0, 35: 1, 36: 1, 37: 0, 38: 0, 39: 0, 40: 1, 41: 0, 42: 1, 44: 0, 46: 0, 47: 1, 48: 0, 49: 0, 50: 0, 51: 0, 52: 1, 53: 1, 54: 0, 55: 0, 56: 0, 57: 0, 58: 0, 59: 0, 60: 0, 61: 0, 62: 0, 63: 0, 65: 0, 66: 0, 67: 0, 68: 1, 69: 1, 71: 0, 72: 0, 73: 0, 74: 0, 76: 0, 78: 1, 79: 1, 80: 0, 81: 1, 82: 0, 83: 0, 84: 0, 85: 0, 87: 0, 88: 0, 89: 0, 90: 1, 91: 0, 92: 0, 93: 0, 94: 0, 95: 0, 96: 0, 98: 0, 99: 0, 100: 0, 101: 1, 102: 0, 103: 1, 104: 0, 105: 0, 106: 0, 108: 1, 112: 0, 113: 0, 114: 0, 115: 0, 118: 0, 119: 0, 120: 1, 121: 1, 122: 0, 123: 0, 124: 0, 125: 1, 126: 0, 127: 0, 128: 0, 129: 1, 131: 1, 132: 0, 133: 0, 134: 0, 135: 0,

In [374]:



scaler_time_elapsed = StandardScaler()
scaler_time_between = StandardScaler()

#Categorical Embeddings
aids_session = []
event_session = []

#Time Embeddings
log_delta_elapsed = []
log_between_time = []

for i, session in enumerate(data_set_after_L):
    single_session = data_set_after_L[i]
    aids_session.extend(single_session["aid"].tolist())
    event_session.extend(single_session["type"].tolist())
    
    
    
    ts_last = single_session["timestamps"][-1]
    ts_first = single_session["timestamps"][0]
    
    log_delta_elapsed.append(log(1 +  (ts_last.item() - ts_first.item())))

    
    deltas_this_session = []
    
    for j in range(len(single_session["timestamps"]) - 1):
        delta_between_times = (single_session["timestamps"][j+1].item() - single_session["timestamps"][j].item())
        deltas_this_session.append(log(1 + delta_between_times))
    log_between_time.append(deltas_this_session)
    


#AID Embeddings
aid_vocab = sorted(set(aids_session))
#aid_to_idx = {aid: i for i, aid in enumerate(aid_vocab)}
num_embeddings_aid = len(aid_vocab)
#Type Event Embeddings
type_event_vocab = sorted(set(event_session))
num_embeddings_event_type = len(type_event_vocab)

#Time Embeddings
array_time_delta_elapsed = numpy.array(log_delta_elapsed).reshape(-1, 1)
standard_time_delta_elapsed = scaler_time_elapsed.fit_transform(array_time_delta_elapsed)

flat_time_between = [d for session_list in log_between_time for d in session_list]
array_time_between = numpy.array(flat_time_between).reshape(-1, 1)
standard_time_between = scaler_time_between.fit_transform(array_time_between)


number_session = 0
session_standard_time_between = []
for session_list in log_between_time:
    L = len(session_list)
    session_standard_time_between.append(standard_time_between[number_session : number_session + L].flatten().tolist())
    number_session += L


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_channels : int, output_channels : int):
        super().__init__()
        self.layers = nn.Sequential(
                nn.Linear(input_channels, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, output_channels)
            )
    def forward(self, x):
            return self.layers(x)
        
        
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.d_model = d_model

        
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)  

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: Tensor) -> Tensor:
        # x: (B, L, d_model)
        x = x + self.pe[:, :x.size(1)]
        return x

In [390]:

class TRACE(nn.Module):
    def __init__(self, num_embeddings_aid : int, 
                 num_embeddings_event_type : int,
                 num_classes : int = 4
               ):
        super(TRACE, self).__init__()
        
        self.D_model = EMBEDDING_DIM * 2 + 2 # 66
        print(self.D_model)
        
        
        self.embedding_aid = nn.Embedding(num_embeddings=num_embeddings_aid,
                                          embedding_dim=EMBEDDING_DIM)
        
        self.embedding_eventtype = nn.Embedding(num_embeddings=num_embeddings_event_type,
                                                embedding_dim=EMBEDDING_DIM) 

        self.positional_embedding = PositionalEncoding(d_model=self.D_model)
        
        #D_model
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=self.D_model, 
                                                        nhead=6, # 8
                                                        dim_feedforward=128,
                                                        dropout=0.2,
                                                        activation="relu",
                                                        batch_first=True)
        
        self.encoder = nn.TransformerEncoder(encoder_layer=self.encoder_layer,
                                             num_layers=1)
        
                
        self.GBAP = nn.AdaptiveMaxPool1d(output_size=1)
        self.__ATC__ = MLP(input_channels=self.D_model, output_channels=1)
        """self.MLP_layer = MLP(input_channels=self.D_model, 
                             output_channels=num_classes)
        """
        
        
    def forward(self, aids_ids: Tensor, type_ids: Tensor, delta_elapsed : Tensor, delta_between : Tensor) -> Tensor:
        
        #Learning Embeddings
      aid_emb = self.embedding_aid(aids_ids)
      type_emb = self.embedding_eventtype(type_ids)
        
      B, L, _ = aid_emb.shape #(Batch, Len_seque , 1)
        
        

      delta_between = delta_between.unsqueeze(-1)        
      delta_between = torch.cat([delta_between, delta_between[:, -1:, :]],dim=1)                                                   # (B, L, 1)


      delta_elapsed = delta_elapsed.unsqueeze(-1).unsqueeze(-1)
      delta_elapsed = delta_elapsed.expand(-1, L, -1)
      

      x = torch.cat(
          [aid_emb,
          type_emb,
          delta_between,
          delta_elapsed],
          dim=-1)  
          
        
      positional_embed = self.positional_embedding(x)
        
      encoder = self.encoder(positional_embed)
        
      global_avarage_pooling = self.GBAP(encoder.transpose(1,2)).squeeze(-1)

      
        
      logits = self.__ATC__(global_avarage_pooling)
        
      return logits  

In [391]:
#DataLoaders
train_loader = DataLoader(dataset=training_data_set, batch_size=32, collate_fn=custom_collate)
testing_loader = DataLoader(dataset=testing_data_set, batch_size=32, collate_fn=custom_collate)

In [392]:
for batch_training in train_loader:
        print(f"Shape of the Training Batch Aids:{batch_training["aid"].shape}, Shape of the Batch Ts:{batch_training["timestamps"].shape}, Shape of the batch Type:{batch_training["events_type"].shape}")
    

print("============================================================================================================================================")
for batch_testing in testing_loader:
    print(f"Shape of the Testing Batch Aids:{batch_testing["aid"].shape}, Shape of the Batch Ts:{batch_testing["timestamps"].shape}, Shape of the batch Type:{batch_testing["events_type"].shape}")
    

Shape of the Training Batch Aids:torch.Size([32, 342]), Shape of the Batch Ts:torch.Size([32, 342]), Shape of the batch Type:torch.Size([32, 342])
Shape of the Training Batch Aids:torch.Size([32, 328]), Shape of the Batch Ts:torch.Size([32, 328]), Shape of the batch Type:torch.Size([32, 328])
Shape of the Training Batch Aids:torch.Size([32, 369]), Shape of the Batch Ts:torch.Size([32, 369]), Shape of the batch Type:torch.Size([32, 369])
Shape of the Training Batch Aids:torch.Size([32, 233]), Shape of the Batch Ts:torch.Size([32, 233]), Shape of the batch Type:torch.Size([32, 233])
Shape of the Training Batch Aids:torch.Size([26, 315]), Shape of the Batch Ts:torch.Size([26, 315]), Shape of the batch Type:torch.Size([26, 315])
Shape of the Testing Batch Aids:torch.Size([32, 65]), Shape of the Batch Ts:torch.Size([32, 65]), Shape of the batch Type:torch.Size([32, 65])
Shape of the Testing Batch Aids:torch.Size([32, 70]), Shape of the Batch Ts:torch.Size([32, 70]), Shape of the batch Type:

C:\Users\alexf\AppData\Local\Temp\ipykernel_13904\1364254648.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  aids = [torch.tensor(d["aid"]) for d in batch]
C:\Users\alexf\AppData\Local\Temp\ipykernel_13904\1364254648.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  timestamps = [torch.tensor(d["timestamps"]) for d in batch]
C:\Users\alexf\AppData\Local\Temp\ipykernel_13904\1364254648.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  events_type = [torch.tensor(d["type"]) for d in batch]


In [393]:
max_aid = max(
    session["aid"].max().item()
    for session in data_set_after_L
)
max_type = max(
    session["type"].max().item()
    for session in data_set_after_L
)



num_embeddings_aid = max_aid + 1  
num_embeddings_event_type = max_type + 1
trace_model = TRACE(
    num_embeddings_aid=num_embeddings_aid,
    num_embeddings_event_type=num_embeddings_event_type,
    num_classes=4
)
"""print("MAX aid in batch:", max(aids))
print("Embedding size:", trace_model.embedding_aid.num_embeddings)
print("MAX type in batch:", len(events_type))
print("Embedding size:", trace_model.embedding_eventtype.num_embeddings)"""

66


'print("MAX aid in batch:", max(aids))\nprint("Embedding size:", trace_model.embedding_aid.num_embeddings)\nprint("MAX type in batch:", len(events_type))\nprint("Embedding size:", trace_model.embedding_eventtype.num_embeddings)'

In [394]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(trace_model.parameters(), lr=1e-5, weight_decay=1e-6)
labels_atc = data_set_after_L.__ATC_task_logit__()

for epoch in range(EPOCHS):
    trace_model.train()
    epoch_loss = 0.0
    for batch_trainer in train_loader:
        aids = batch_trainer["aid"].long() 
        sessions_id = [x.item() for x in batch_trainer["session_id"]]
        
        
        labels_for_batch = [labels_atc[id] for id in sessions_id]
        labels_for_atc_tensor = torch.tensor(labels_for_batch, dtype=torch.float32).unsqueeze(-1)
        
            
        events_type = batch_trainer["events_type"].long()
        timestamps = batch_trainer["timestamps"]      

        ts_first = timestamps[:, 0]
        ts_last = timestamps[:, -1]

        
        log_delta_elapsed = torch.log1p(torch.clamp(ts_last - ts_first, min=0))

        
        delta_elapsed = (log_delta_elapsed - log_delta_elapsed.mean()) / (log_delta_elapsed.std() + 1e-6)           # (B,)

    
        log_delta_between = torch.log1p(torch.clamp(timestamps[:, 1:] - timestamps[:, :-1], min=0))

        mean_between = log_delta_between.mean(dim=1, keepdim=True)
        std_between  = log_delta_between.std(dim=1, keepdim=True)

        delta_between = (log_delta_between - mean_between) / (std_between + 1e-6)                        

    
        model = trace_model(
            aids,
            events_type,
            delta_elapsed,
            delta_between
        )

        loss = criterion(model, labels_for_atc_tensor)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    train_loss = epoch_loss/len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {epoch_loss/len(train_loader):.4f}")


C:\Users\alexf\AppData\Local\Temp\ipykernel_13904\1364254648.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  aids = [torch.tensor(d["aid"]) for d in batch]
C:\Users\alexf\AppData\Local\Temp\ipykernel_13904\1364254648.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  timestamps = [torch.tensor(d["timestamps"]) for d in batch]
C:\Users\alexf\AppData\Local\Temp\ipykernel_13904\1364254648.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  events_type = [torch.tensor(d["type"]) for d in batch]


Epoch [1/10], Loss: 0.6646
Epoch [2/10], Loss: 0.6637
Epoch [3/10], Loss: 0.6622
Epoch [4/10], Loss: 0.6615
Epoch [5/10], Loss: 0.6597
Epoch [6/10], Loss: 0.6584
Epoch [7/10], Loss: 0.6574
Epoch [8/10], Loss: 0.6559
Epoch [9/10], Loss: 0.6543
Epoch [10/10], Loss: 0.6531


In [321]:
import torch
import torch.nn as nn

# ====== Configuración ======
BATCH = 16
L = 48
EMBEDDING_DIM = 32
num_embeddings_aid = 5000
num_embeddings_event_type = 10
num_classes = 4

# ====== Modelo ======
trace_model = TRACE(
    num_embeddings_aid=num_embeddings_aid,
    num_embeddings_event_type=num_embeddings_event_type,
    num_classes=num_classes
)

criterion = nn.CrossEntropyLoss()

# ====== Mock batch CORRECTO ======
aids_ids = torch.randint(0, num_embeddings_aid, (BATCH, L))      # (B, L)
type_ids = torch.randint(0, num_embeddings_event_type, (BATCH, L))  # (B, L)

delta_elapsed = torch.randn(BATCH)       # ✅ (B,)
delta_between = torch.randn(BATCH, L-1)  # ✅ (B, L-1)

labels = torch.randint(0, num_classes, (BATCH,))  # (B,)

# ====== Forward ======
logits = trace_model(aids_ids, type_ids, delta_elapsed, delta_between)

print("Logits shape:", logits.shape)   # ✅ debe ser (BATCH, 4)

loss = criterion(logits, labels)
print("Loss:", loss.item())


66
torch.Size([16, 66])
Logits shape: torch.Size([16, 4])
Loss: 1.4645079374313354
